# Instalando Bibliotecas


In [ ]:
%pip install -qU pypdf
%pip install -U langchain
%pip install -U langchain-community
%pip install -U langchain-groq
%pip install langchain-huggingface
%pip install langgraph


# Conectando API KEY


In [ ]:
from google.colab import userdata
import os
api_key=userdata.get('GROQ_API_KEY')
os.environ['GROQ_API_KEY'] = api_key

#Importando Chat Groq do LangChain

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(model='llama-3.3-70b-versatile')

In [ ]:
prompt = '''
	messages: Segundo a NASA quais seriam os benefícios científicos de ir para Marte?
'''

In [ ]:
resposta = llm.invoke(prompt)

In [ ]:
resposta

# Adicionando documento pdf para análise

In [ ]:
url = 'https://raw.githubusercontent.com/arletedelira-TI/analise-llm-projeto-nasa/main/document-nasa-s-moon-to-mars-strategy-and-objectives-development.pdf'

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(url)
pages = []
for page in loader.lazy_load():
	pages.append(page)

In [ ]:
print(f"{pages[0].metadata}\n")

In [ ]:
print(pages[0].page_content)

# Criando uma base de dados vetorial

## Obs: Lembre-se de habilitar a opção GPU no google colab

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
embed_model = HuggingFaceEmbeddings (model_name="mixedbread-ai/mxbai-embed-large-v1")

In [ ]:
vector_store = InMemoryVectorStore.from_documents(pages, embed_model)

# Gerando uma resposta com RAG

In [ ]:
docs = vector_store.similarity_search("Objective Development Process, k=2")

In [ ]:
for doc in docs:
	print(f'Page {doc.metadata["page"]}: {doc.page_content[:300]}\n')

# Pipeline de RAG

In [ ]:
retriever = vector_store.as_retriever()

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
template = """ You're a helpful assistant that only gives answers bases on the given context. If the answer is not in the context, say "I don't know".

Context: {context}

Question: {question}

Answer:

"""

In [ ]:
prompt = ChatPromptTemplate.from_template(template)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
from IPython.display import display, Markdown

In [ ]:
response = chain.invoke("What are the Objectives Development Process?")
display(Markdown(response))